# Lecture 8. Yes/no variables and the linear probability model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture08/lecture08.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Wooldridge, *Introductory Econometrics*, sections 7-1 to 7-5;
section 7-6 as background.

Today we use the routine from Lecture 7 (fit, check, fix, extend, check again) in two ways:

| Part | What we do | Step in the routine |
|---|---|---|
| 1 | Add a yes/no variable, club membership, to the final model of Lecture 7 | Extend |
| 2 | Add a category with more than two values, region | Extend |
| 3 | Let the membership gap depend on tenure: an interaction | Extend, then check |
| 4 | A new outcome that is 1 or 0, churn: the linear probability model | Fit, check, fix, check again |

In Parts 1 to 3 the outcome stays ln spend, so everything from Lecture 7 still applies. Part 4 has a new outcome, so
it starts the routine again. Where Python does something differently from Wooldridge, a section "... in Python and
in Wooldridge" says so. Each part ends with an exercise: you predict a result, run the cell, change one value, and
answer a question. You do not need to write any code. The last section holds a summary table, the symbols and the
Danish terms.

Run the setup cell first. It loads the RetailCorp customer data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 customers drawn at random, as in Lectures 6 and 7
customers["log_spend"] = np.log(customers["spend_2025"])

print(f"Customers: {len(customers):,}")
customers.head()

## The case and the question

The same 1,000 RetailCorp customers as in Lectures 6 and 7. Today we add these columns:

| Column | Content | Used in |
|---|---|---|
| `member` | 1 if the customer was in the loyalty club on 1 January 2025, else 0 | Part 1 |
| `channel` | The customer's main channel in 2025: App or Web | Exercise 1, Part 3 |
| `region` | One of the five Danish regions | Part 2 |
| `churned` | 1 if the customer placed no order in January to June 2026, else 0 | Part 4 (the outcome) |

**Lead question.** What do yes/no facts about a customer tell us about how much they spend, and about who stops
buying?

A **dummy variable** is 1 for the customers in a group and 0 for the others. The group coded 0 is the one the others
are compared with. A category with more than two values needs several dummies. Dummy variables are also called
indicator or binary variables; Wooldridge calls them qualitative information (7-1).

## Part 1. One yes/no variable (Extend)

**Case question.** Do club members spend more than other customers, and could the difference be chance?

We start with spend in DKK, so the link to Lecture 4 is easy to see, and then return to ln spend, the outcome from
Lecture 7.

### Two group means

In [ ]:
means = customers.groupby("member")["spend_2025"].mean()
counts = customers.groupby("member").size()
print(means.round(2))
print(counts)
print(f"Difference: {means[1] - means[0]:.2f}")

### A regression on one dummy

The model is y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> + ε<sub>i</sub>, read "y-i equals
beta-zero plus beta-one times member-i plus epsilon-i".

- For non-members (member = 0) the average is β<sub>0</sub>.
- For members (member = 1) the average is β<sub>0</sub> + β<sub>1</sub>.

So x takes only two values, and the slope is the step from one group to the other. Least squares puts the line
through the two group means: b<sub>0</sub> is the mean of non-members, and b<sub>1</sub> is the difference between
the two means.

**By hand.** b<sub>0</sub> = 1,849.21; b<sub>1</sub> = 2,622.48 − 1,849.21 = 773.27; t = 773.27 / 93.21 = 8.30 with
n − 2 = 998 degrees of freedom (critical value 1.9623).

In [ ]:
m1 = smf.ols("spend_2025 ~ member", data=customers).fit()
print(m1.params.round(2))
print(f"se = {m1.bse['member']:.2f}, t = {m1.tvalues['member']:.2f}")

**The test from Lecture 4.** The pooled two-sample t test of members against non-members gives the same t. With
robust standard errors (HC1) the regression gives z = 8.10, close to Welch's t (`equal_var=False`), the version
that does not assume equal variances.

In [ ]:
spend_m = customers.loc[customers["member"] == 1, "spend_2025"]
spend_n = customers.loc[customers["member"] == 0, "spend_2025"]
print(f"Pooled t: {stats.ttest_ind(spend_m, spend_n).statistic:.2f}")
print(f"Welch t:  {stats.ttest_ind(spend_m, spend_n, equal_var=False).statistic:.2f}")

m1_r = smf.ols("spend_2025 ~ member", data=customers).fit(cov_type="HC1")
print(f"Regression with HC1: z = {m1_r.tvalues['member']:.2f}")

**For the manager.** In 2025, club members spent about 773 DKK more than other customers on average (2,622 against
1,849 DKK). A difference this large would be very unlikely if members and non-members spent the same on average.

### Membership in ln spend

As in Lecture 7, spend is skewed, and we model ln spend with robust standard errors. A dummy in a model for ln y is
read like any x in Lecture 7: about 100 × b percent, exactly 100 × (exp(b) − 1) percent.

In [ ]:
m_alone = smf.ols("log_spend ~ member", data=customers).fit(cov_type="HC1")
b = m_alone.params["member"]
print(m_alone.params.round(4))
print(f"About {100 * b:.1f}%, exactly {100 * (np.exp(b) - 1):.1f}%")

**The number e.** e ≈ 2.718 (Euler's number) is the base of the natural log. ln and the exponential function exp undo
each other: `np.log` and `np.exp`. A difference of b in ln spend means spend is multiplied by e to the power b,
exp(b). Here exp(0.4022) = 1.495, so members spend 49.5% more. The about version is close only for small b, below
about 0.1:

In [ ]:
for b in [0.0397, 0.2197, 0.4022]:
    print(f"b = {b:.4f}: about {100 * b:.1f}%, exactly {100 * (np.exp(b) - 1):.1f}%")

### Membership with controls

The final model from Lecture 7 with member added:

ln y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> + β<sub>2</sub> tenure<sub>i</sub> +
β<sub>3</sub> age<sub>i</sub> + β<sub>4</sub> visits<sub>i</sub> + ε<sub>i</sub>

Member is now the **variable of interest**, and tenure, age and visits are **controls**. The order of the x's in the
formula does not matter, and every coefficient compares customers who are alike on the other x's. The roles come
from the question: we ask about member, and the controls are there because members differ on them and they go with
spend. In Lecture 7, tenure was the variable of interest.

The cell adds the controls one step at a time and prints the member coefficient.

In [ ]:
formulas = ["log_spend ~ member",
            "log_spend ~ member + tenure_years + age",
            "log_spend ~ member + tenure_years + age + visits_2025"]
for f in formulas:
    m_ = smf.ols(f, data=customers).fit(cov_type="HC1")
    b = m_.params["member"]
    print(f"{f}:  member = {b:.4f}, exactly {100 * (np.exp(b) - 1):.1f}%")

In [ ]:
m2 = smf.ols("log_spend ~ member + tenure_years + age + visits_2025", data=customers).fit(cov_type="HC1")
low, high = m2.conf_int().loc["member"]
print(f"Member, exactly: {100 * (np.exp(m2.params['member']) - 1):.1f}%, "
      f"95% interval {100 * (np.exp(low) - 1):.1f}% to {100 * (np.exp(high) - 1):.1f}%")

# Why the coefficient shrinks: the group averages
print(customers.groupby("member")[["tenure_years", "visits_2025", "age"]].mean().round(2))

Members have been customers longer (3.87 against 2.93 years), visit more (34.4 against 26.3) and are slightly older
(48.3 against 45.5), and all three go with higher spend. So part of the raw gap of 49.5% was a tenure, visits and age
difference. As in Lecture 7, a variable related
to both x and y changes the coefficient when it is added.

**Read the controls' coefficients with care.** The model was built to compare members and non-members, so a control's
coefficient can still carry other left-out differences. Report and interpret the variable of interest, and treat the
controls as part of the setup.

**Association.** Customers chose to join the club, so 24.6% compares members and non-members as they are and says
nothing about what joining would do to a customer's spend (Wooldridge 7-6 calls this self-selection).

### Dummies in Python and in Wooldridge

| Point | Wooldridge 7-1, 7-2 | statsmodels |
|---|---|---|
| A 0/1 variable | Named after the group coded 1 (female, married) | Enters as it is: `member` |
| A text variable | Turned into 0/1 variables by the analyst | Turned into dummies in the formula; the first value in alphabetical order is the group coded 0: `C(channel)[T.Web]` |
| Exact percent for ln y | 100 · [exp(b) − 1] (7-2a) | `100 * (np.exp(b) - 1)`, as in Lecture 7 |
| Symbols | δ<sub>0</sub> (delta-zero) for a dummy's coefficient, u for the error | We keep β and ε |

`C(channel)[T.Web]` reads as: channel, with Web compared with the other value, App. T stands for treatment coding,
the default. You can write `channel` or `C(channel)` in the formula; C() makes it clear that the column is a category.

In [ ]:
m_ch = smf.ols("log_spend ~ C(channel)", data=customers).fit(cov_type="HC1")
print(m_ch.params.round(4))

### Exercise 1. Channel in place of membership

**Predict.** App users are younger, newer customers and visit more. Do web customers spend less than app users, the same, or more?
Without controls, and with tenure, age and visits?

**Run** the cell. It fits ln spend on channel alone and with the three controls, with robust standard errors.

In [ ]:
controls = "tenure_years + age + visits_2025"     # change to "tenure_years + age" in step 3

for f in ["log_spend ~ C(channel)", f"log_spend ~ C(channel) + {controls}"]:
    m_ = smf.ols(f, data=customers).fit(cov_type="HC1")
    b = m_.params["C(channel)[T.Web]"]
    p = m_.pvalues["C(channel)[T.Web]"]
    print(f"{f}")
    print(f"   Web: {b:.4f}, exactly {100 * (np.exp(b) - 1):.2f}%, p = {p:.3f}")

print(customers.groupby("channel")[["age", "tenure_years", "visits_2025"]].mean().round(2))

**Change** `controls` to `"tenure_years + age"` (visits left out) and run the cell again.

**Answer.** How does the Web coefficient change from one model to the next, and why?

## Part 2. More than two categories (Extend)

**Case question.** Does spend differ between regions?

One dummy per region gives five dummies, and for every customer exactly one of them is 1:

Hovedstaden<sub>i</sub> + Midtjylland<sub>i</sub> + Nordjylland<sub>i</sub> + Sjælland<sub>i</sub> +
Syddanmark<sub>i</sub> = 1

The five dummies add up to the column of ones behind the intercept, so the model cannot tell the intercept apart from
them. This is perfect multicollinearity; Wooldridge calls it the dummy variable trap. So we use k − 1 = 4 dummies for
k = 5 regions. The region left out is the **reference category**, and every region coefficient is a difference from
it. statsmodels leaves out the first region in alphabetical order, Hovedstaden.

In [ ]:
print(customers["region"].value_counts())

In [ ]:
m_reg = smf.ols("log_spend ~ member + C(region) + tenure_years + age + visits_2025",
                data=customers).fit(cov_type="HC1")
for name in m_reg.params.index:
    if name.startswith("C(region)") or name == "member":
        b = m_reg.params[name]
        print(f"{name:26s} {b:8.4f}   p = {m_reg.pvalues[name]:.3f}   exactly {100 * (np.exp(b) - 1):6.1f}%")
print(f"Intercept = {m_reg.params['Intercept']:.4f}, R2 = {m_reg.rsquared:.4f}")

Each row compares a region with Hovedstaden, at the same membership, tenure, age and visits. Customers in Nordjylland
spend about 9.5% less than in Hovedstaden (p = 0.088). No region differs clearly from Hovedstaden. Each p-value
answers whether one region differs from Hovedstaden; whether region matters at all is a joint question about the four
coefficients together. The member coefficient hardly moves (0.2197 without region, 0.2169 with).

### Choosing the reference category

`C(region, Treatment('Nordjylland'))` makes Nordjylland the reference. The intercept, the region coefficients and
their p-values change, because each is now a difference from Nordjylland. The predictions and residuals, R², adjusted
R² and the other coefficients stay the same: it is the same model written another way. Choose the reference that
makes the comparison you want to report, often the largest group or a natural baseline, and say which it is.

Region is left out from here, since it changes the other coefficients very little.

### The trap in Python and in Wooldridge

`east` is 1 for Hovedstaden and Sjælland. It can be written from the region dummies, so the model cannot separate it
from them. Wooldridge (7-3) notes that some programs drop a variable for you and others report perfect collinearity.
statsmodels prints coefficients anyway, here without meaning, with standard errors and p-values that look ordinary.
Read the notes under every output: a note on a tiny smallest eigenvalue, like [2] here, means one x can be written
from the others. Newer versions of statsmodels also print a warning above the output.

In [ ]:
customers["east"] = customers["region"].isin(["Hovedstaden", "Sjælland"]).astype(int)
m_e = smf.ols("log_spend ~ member + C(region) + east"
              " + tenure_years + age + visits_2025",
              data=customers).fit()
print(m_e.summary())

### Exercise 2. Another reference category

**Predict.** The cell refits the region model with Nordjylland as the reference. Which numbers change: the intercept,
the region coefficients, their p-values, the member coefficient, R²?

**Run** the cell and compare with the table with Hovedstaden as the reference above.

In [ ]:
ref = "Nordjylland"     # change to "Midtjylland" in step 3

m_ref = smf.ols(f"log_spend ~ member + C(region, Treatment('{ref}')) + tenure_years + age + visits_2025",
                data=customers).fit(cov_type="HC1")
for name in m_ref.params.index:
    if name.startswith("C(region") or name in ("Intercept", "member"):
        print(f"{name:50s} {m_ref.params[name]:8.4f}   p = {m_ref.pvalues[name]:.3f}")
print(f"R2 = {m_ref.rsquared:.4f}")

**Change** `ref = "Nordjylland"` to `"Midtjylland"` and run the cell again.

**Answer.** What is the difference between Hovedstaden and Nordjylland in each version? Why does R² stay the same?

## Part 3. Interactions (Extend)

**Case question.** Is the spend gap between members and non-members the same for new and long-standing customers?

At the same tenure, age and visits, members spend 24.6% more. That model has one gap for all customers, whatever
their tenure: in a picture, members and non-members have parallel lines against tenure. Wooldridge 7-4 calls the
alternative allowing for different slopes.

### A model with an interaction

ln y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> + β<sub>2</sub> tenure<sub>i</sub> +
β<sub>3</sub> (member<sub>i</sub> × tenure<sub>i</sub>) + … + ε<sub>i</sub>

The interaction term member × tenure is 0 for non-members and the tenure for members. At the same age and visits:

- non-members: β<sub>0</sub> + β<sub>2</sub> tenure
- members: (β<sub>0</sub> + β<sub>1</sub>) + (β<sub>2</sub> + β<sub>3</sub>) tenure

So the gap at a given tenure is β<sub>1</sub> + β<sub>3</sub> × tenure. β<sub>1</sub> is the gap at tenure 0, and
β<sub>3</sub> is how much the gap changes per year of tenure. β<sub>3</sub> = 0 means the gap does not change with
tenure. "The gap depends on tenure" and "the tenure slope depends on membership" are the same statement; both
describe β<sub>3</sub>.

Note that β<sub>1</sub> changes meaning: it is now the gap for a new customer, so it cannot be compared with 0.2197
from Part 1.

### The interaction in Python

`a * b` in the formula gives a, b and their product a:b.

In [ ]:
m_int = smf.ols("log_spend ~ member * tenure_years + age + visits_2025", data=customers).fit(cov_type="HC1")
keep = ["member", "tenure_years", "member:tenure_years"]
print(m_int.params[keep].round(4))

b = m_int.params
gap = b["member"] + b["member:tenure_years"] * 3.3
print(f"{100 * (np.exp(gap) - 1):.1f}")

**By hand.**

1. Gap for a new customer (tenure 0): 100 × (e<sup>0.4117</sup> − 1) = 50.9%.
2. Gap at the average tenure of 3.3 years: 0.4117 − 0.0561 × 3.3 = 0.2266, and 100 × (e<sup>0.2266</sup> − 1) = 25.4%.
3. Where the gap reaches 0: 0.4117 / 0.0561 = 7.3 years.
4. Test of H<sub>0</sub>: β<sub>3</sub> = 0, with robust standard errors, so z: z = −0.0561 / 0.01278 = −4.39 < −1.96,
   so we reject H<sub>0</sub>.

In [ ]:
print(f"z for the interaction: {m_int.tvalues['member:tenure_years']:.2f}, "
      f"p = {m_int.pvalues['member:tenure_years']:.1e}")

### The gap across tenure

The figure shows predicted spend from the interaction model at the average age and visits, taken back to DKK with
exp. The lines curve slightly because spend is exp of a straight line in ln spend.

In [ ]:
tenure_grid = np.linspace(0, 10, 41)
plt.figure(figsize=(7, 4))
for member, label in [(0, "Non-members"), (1, "Members")]:
    new = pd.DataFrame({"member": member, "tenure_years": tenure_grid,
                        "age": customers["age"].mean(), "visits_2025": customers["visits_2025"].mean()})
    plt.plot(tenure_grid, np.exp(m_int.predict(new)), label=label)
plt.axvline(-m_int.params["member"] / m_int.params["member:tenure_years"], color="gray", linestyle="--")
plt.xlabel("Years as a customer")
plt.ylabel("Predicted spend, DKK")
plt.legend()
plt.show()

print(f"Customers with more than 7 years: {(customers['tenure_years'] > 7).sum()}")

New members spend about 51% more than new non-members. The gap narrows with each year, and the lines meet after
about 7 years. Beyond that the lines rest on few customers (111 of the 1,000 have more than 7 years), so do not read
much into the members' line falling below.

**Check again.** The same checks as in Lecture 7, on the model with the interaction:

In [ ]:
m_int_c = smf.ols("log_spend ~ member * tenure_years + age + visits_2025", data=customers).fit()
lm, lm_p, f, f_p = het_breuschpagan(m_int_c.resid, m_int_c.model.exog)
jb, jb_p = stats.jarque_bera(m_int_c.resid)
print(f"Breusch-Pagan = {lm:.2f}, p = {lm_p:.2f}")
print(f"Skew = {stats.skew(m_int_c.resid):.2f}, Kurtosis = {stats.kurtosis(m_int_c.resid, fisher=False):.2f}, "
      f"Prob(JB) = {jb_p:.2f}")

No fan, and the residuals are close to normal.

**For the manager.** Among customers of the same age and with the same number of visits, new members spend about
50% more than new non-members. The gap narrows with each year as a customer and is close to zero after about seven
years. Customers choose to join, so this describes customers as they are.

### Interaction of two yes/no variables

When the second variable is also yes/no, there are no slopes, just four groups. With
ln y<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> + β<sub>2</sub> web<sub>i</sub> +
β<sub>3</sub> (member<sub>i</sub> × web<sub>i</sub>) + … + ε<sub>i</sub>, the average ln spend in each group, at the
same tenure, age and visits, is:

| | App (web = 0) | Web (web = 1) |
|---|---|---|
| Non-members | β<sub>0</sub> | β<sub>0</sub> + β<sub>2</sub> |
| Members | β<sub>0</sub> + β<sub>1</sub> | β<sub>0</sub> + β<sub>1</sub> + β<sub>2</sub> + β<sub>3</sub> |
| Gap, members minus non-members | β<sub>1</sub> | β<sub>1</sub> + β<sub>3</sub> |

In words: the gap for app users is β<sub>1</sub>, the gap for web users is β<sub>1</sub> + β<sub>3</sub>, and
β<sub>3</sub> is the difference between the two gaps. β<sub>3</sub> = 0 means the gap is the same in both channels.
`web` is the dummy statsmodels makes from channel, `C(channel)[T.Web]`.

### Exercise 3. Is the club gap the same for app and web users?

**Predict.** Will the membership gap be larger for app users, for web users, or about the same?

**Run** the cell. It fits ln spend on member * C(channel) with tenure, age and visits, and robust standard errors.

In [ ]:
ref = "App"     # change to "Web" in step 3

m_e3 = smf.ols(f"log_spend ~ member * C(channel, Treatment('{ref}')) + tenure_years + age + visits_2025",
               data=customers).fit(cov_type="HC1")
names = [n for n in m_e3.params.index if n.startswith("member")]
for name in names:
    print(f"{name:45s} {m_e3.params[name]:8.4f}   p = {m_e3.pvalues[name]:.3f}")

gap_ref = m_e3.params[names[0]]
gap_other = m_e3.params[names[0]] + m_e3.params[names[1]]
print(f"Gap for {ref} users:       exactly {100 * (np.exp(gap_ref) - 1):.1f}%")
print(f"Gap for the other channel: exactly {100 * (np.exp(gap_other) - 1):.1f}%")

**Change** `ref = "App"` to `"Web"`, so App becomes the dummy, and run the cell again.

**Answer.** What is the gap for app users and for web users, in percent? Could the difference between them be
chance? What changed when you set `ref`?

### Interactions in Python and in Wooldridge

| In the formula | What goes into the model | When to use it |
|---|---|---|
| `member * tenure_years` | member, tenure_years and member:tenure_years | The usual case: both main terms and their product |
| `member:tenure_years` | Only the product | When the main term is already in the model in another form |
| `member + C(region) + member:east` | member, four region dummies, and the product | east is already in the region dummies (Part 2), so only the product is added |

Wooldridge 7-4 makes the product as a new variable (for example female · educ) and keeps both main terms. Without a
main term, β<sub>3</sub> mixes the gap at tenure 0 with its change per year.

In [ ]:
m_east = smf.ols("log_spend ~ member + C(region) + member:east + tenure_years + age + visits_2025",
                 data=customers).fit(cov_type="HC1")
print(m_east.params.round(4))

Here the member coefficient is the club gap for customers outside the east, and `member:east` is how much the gap
differs in Hovedstaden and Sjælland. `east` itself is not added, because the region dummies already contain it.

## Part 4. A yes/no outcome (Fit, check, fix, check again)

### A new outcome: churn

`churned` is 1 if the customer placed no order in January to June 2026, and 0 otherwise. The mean of a 0/1 variable
is the share of 1s, the proportion from Lecture 3.

In [ ]:
print(f"Share churned: {customers['churned'].mean():.3f}")
print(customers.groupby("member")["churned"].agg(["count", "sum", "mean"]).round(3))

Members churned less (17.8% against 36.6%), but they also have longer tenure and more visits. The model compares
customers alike on those.

**Case question.** Which customers stop buying, and how much lower is churn for members at the same tenure, age and
visits?

### Fit: the linear probability model

With a 0/1 outcome, least squares gives the linear probability model, LPM (Wooldridge 7-5):

churned<sub>i</sub> = β<sub>0</sub> + β<sub>1</sub> member<sub>i</sub> + β<sub>2</sub> web<sub>i</sub> +
β<sub>3</sub> tenure<sub>i</sub> + β<sub>4</sub> age<sub>i</sub> + β<sub>5</sub> visits<sub>i</sub> +
ε<sub>i</sub>

The fitted value ŷ<sub>i</sub> = P̂(churned<sub>i</sub> = 1) is the predicted probability that customer i churns.
Coefficients are differences in a probability. Times 100 they are **percentage points**: b<sub>1</sub> = −0.10 means
10 percentage points lower. Nothing in the code is new: a 0/1 column works as y like any other, and `predict` gives
predicted probabilities.

In [ ]:
m_lpm = smf.ols("churned ~ member + C(channel) + tenure_years"
                " + age + visits_2025",
                data=customers).fit(cov_type="HC1")
print(m_lpm.params.round(4))

new = pd.DataFrame({"member": [0, 1], "channel": "Web",
                    "tenure_years": 2, "age": 46, "visits_2025": 24})
print(m_lpm.predict(new).round(2))

**By hand.** A web customer who is not a member, with 2 years, aged 46 and 24 visits:
0.6264 + 0.1011 − 0.0260 × 2 − 0.0015 × 46 − 0.0060 × 24 = 0.46. The same customer as a member: 0.46 − 0.11 = 0.35.
Members: 10.7 percentage points lower churn at the same channel, tenure, age and visits. Web customers: about 10
points higher than app users. One more year as a customer: 2.6 points lower.

**For the manager.** Comparing customers with the same channel, tenure, age and visits, members were about 11
percentage points less likely to stop buying in the first half of 2026 (95% interval 5 to 16 points). Web customers
were about 10 points more likely to stop than app users.

In [ ]:
low, high = m_lpm.conf_int().loc["member"]
print(f"member: {100 * m_lpm.params['member']:.1f} points, 95% interval {100 * low:.1f} to {100 * high:.1f}")

### Check: the residual plot

y is 0 or 1, so each residual is either 1 − ŷ (churned) or −ŷ (did not churn). The points lie on two bands. That is
what every LPM residual plot looks like.

The scatter cannot be the same for all customers: a 0/1 outcome with probability p<sub>i</sub> has variance
Var(ε<sub>i</sub>) = p<sub>i</sub>(1 − p<sub>i</sub>), read "p-i times one minus p-i". It is largest at 0.5 and small
near 0 and 1, and the probability differs between customers. Breusch-Pagan confirms it. Normality checks do not apply:
the residuals take two values for each ŷ.

In [ ]:
m_lpm_c = smf.ols("churned ~ member + C(channel) + tenure_years + age + visits_2025", data=customers).fit()

plt.figure(figsize=(7, 4))
plt.scatter(m_lpm_c.fittedvalues, m_lpm_c.resid, s=8, alpha=0.5)
plt.axhline(0, color="darkorange", linestyle="--")
plt.xlabel("Fitted value: the predicted probability")
plt.ylabel("Residual")
plt.show()

lm, lm_p, f, f_p = het_breuschpagan(m_lpm_c.resid, m_lpm_c.model.exog)
print(f"Breusch-Pagan = {lm:.1f}, p = {lm_p:.1e}")

### Fix: robust standard errors, always

Since unequal scatter is built into the model, the classical standard errors rest on an assumption that cannot hold.
Here the two kinds are close, but you cannot know that without computing both. Use robust standard errors for every
LPM. Wooldridge 7-5 says the LPM must be heteroskedastic, notes that the difference is often small in practice, and
corrects the standard errors in chapter 8; statsmodels
gives classical ones unless you add `cov_type="HC1"`. The coefficients are the same with both kinds of standard
errors; only the standard errors differ.

In [ ]:
table = pd.DataFrame({"coef": m_lpm.params,
                      "classical se": m_lpm_c.bse,
                      "robust se (HC1)": m_lpm.bse})
print(table.round(4))

### Check again: the predicted probabilities

In [ ]:
p_hat = m_lpm_c.fittedvalues
print(f"Below 0: {(p_hat < 0).sum()}, above 1: {(p_hat > 1).sum()}")
print(f"Lowest: {p_hat.min():.2f}, highest: {p_hat.max():.2f}")

below = customers[p_hat < 0]
print(f"Members among them: {below['member'].sum()} of {len(below)}")
print(f"Their average visits and tenure: {below['visits_2025'].mean():.0f} and {below['tenure_years'].mean():.1f}")
print(f"All customers: {customers['visits_2025'].mean():.0f} and {customers['tenure_years'].mean():.1f}")

plt.figure(figsize=(7, 3.5))
plt.hist(p_hat, bins=np.arange(-0.25, 0.701, 0.05), edgecolor="white")
plt.axvline(0, color="darkorange", linestyle="--")
plt.xlabel("Predicted probability of churn")
plt.ylabel("Customers")
plt.show()

41 predicted probabilities are below 0. Most of them are members with many visits and long tenure. A straight line
keeps falling as visits and tenure grow, while a probability cannot go below 0. For most customers the predictions
are usable. Lecture 9 uses curves that stay between 0 and 1.

### Percent and percentage points

| Comparison | Difference in percentage points | Difference in percent |
|---|---|---|
| Share churned: non-members 36.6%, members 17.8% | 36.6 − 17.8: about 19 points lower for members | About 19 / 36.6: about 51% lower, about half |
| LPM coefficient on member, −0.1075 | 10.7 points lower, at the same channel, tenure, age and visits | Depends on the starting level, so not given by the coefficient |
| ln spend coefficient on member, 0.2197 | Not a share, so no points | Exactly 24.6% higher spend (Part 1) |

A percentage point is the difference between two percentages. A percent is a change relative to the starting level.
An LPM coefficient is always in points; a dummy in a model for ln y is in percent.

### Exercise 4. Reading the linear probability model

The output below is the LPM with robust standard errors (it is also on the slide). With pen and paper:

1. **Read** the member coefficient in percentage points, with its 95% interval.
2. **Compute** the predicted probability of churn for a member using the app, with 8 years, aged 50, with 60 visits.
3. **Explain** what is wrong with the result, and what it says about the straight line.
4. **Correct.** A colleague writes: "members churn 10.7% less". Rewrite the sentence.

In [ ]:
print(m_lpm.summary())

Run this cell to check your prediction in step 2.

In [ ]:
customer = pd.DataFrame({"member": [1], "channel": ["App"], "tenure_years": [8], "age": [50], "visits_2025": [60]})
print(round(m_lpm.predict(customer).iloc[0], 4))

## Reporting with yes/no variables

We regressed ln spend in 2025 on club membership, years as a customer, age and visits for a random sample of 1,000
customers (OLS, robust standard errors HC1). Comparing customers with the same tenure, age and visits, members spent
24.6% more than non-members (95% interval 16.4% to 33.4%). The gap is about 51% for new customers and narrows with
tenure (interaction p < 0.001). In a linear probability model for churn in the first half of 2026, members were 10.7
percentage points less likely to churn (95% interval 5.2 to 16.3), at the same channel, tenure, age and visits; 41 of
the 1,000 predicted probabilities are below 0. Customers choose to join the club, so these are differences between
customers as they are.

Report: the reference category of every categorical x; percent for ln y and percentage points for a 0/1 outcome; for
an interaction, the gap at a few values the reader cares about; robust standard errors, and fitted values outside 0
to 1 for an LPM.

## Reference

### Reading coefficients in one table

| Outcome y | x is a number (one unit more) | x is a dummy (the group against the reference) | Interaction with a dummy |
|---|---|---|---|
| y in DKK | b DKK | b DKK | b<sub>3</sub>: the change in the DKK gap per unit of x |
| ln y | About 100 · b percent, exactly 100 · (exp(b) − 1) | The same: exactly 100 · (exp(b) − 1) percent | The gap at x is b<sub>1</sub> + b<sub>3</sub> · x on the ln scale; take exp to get percent |
| y is 0 or 1 (LPM) | 100 · b percentage points | 100 · b percentage points | 100 · b<sub>3</sub>: the change in the gap in points per unit of x |

Every coefficient compares customers who are alike on the other x's in the model, and a dummy's coefficient compares
with the reference category.

### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| member<sub>i</sub> | member-i | 1 if customer i is a club member, 0 otherwise |
| β<sub>0</sub> | beta-zero | With one dummy: the average of the group coded 0 |
| β<sub>1</sub> | beta-one | With one dummy: the difference between the groups; with an interaction: the gap at tenure 0 |
| β<sub>3</sub> | beta-three | With an interaction: how much the gap changes per unit of the other x |
| e | e | Euler's number, about 2.718; `np.exp(b)` is e to the power b |
| ŷ<sub>i</sub> | y-hat-i | In an LPM: the predicted probability that customer i has y = 1 |
| P̂(churned<sub>i</sub> = 1) | P-hat of churned equals 1 | The same, written as a probability |
| p<sub>i</sub>(1 − p<sub>i</sub>) | p-i times one minus p-i | The variance of a 0/1 outcome with probability p<sub>i</sub> |

### Danish terms

| English | Danish |
|---|---|
| dummy variable | dummyvariabel |
| categorical variable | kategorisk variabel |
| reference category | referencekategori |
| perfect multicollinearity | perfekt multikollinearitet |
| interaction term | interaktionsled |
| interaction | vekselvirkning |
| linear probability model | lineær sandsynlighedsmodel |
| percentage point | procentpoint |